# ArcFace — huấn luyện lại S3 (Nôm) trên Kaggle · reset-safe

Chạy **tuần tự** các cell dưới. Nếu session Kaggle bị reset/hết giờ → **chạy lại cả notebook**,
cell TRAIN sẽ tự **resume từ Hugging Face** (không train lại từ đầu).

**Trước khi chạy:** Settings → GPU **T4 x2** + **Internet ON** · Add-ons → Secrets → thêm `HF_TOKEN`
(quyền write) · Add Data → gắn dataset chứa `ArcFace/data/` (đã `prepare_data.py` ở máy) và code `ArcFace/*.py`.

### 1) Cấu hình — sửa ở đây

In [ ]:
HF_REPO = ""            # ví dụ "mdnt571/nom-embed-arcface"  ← BẮT BUỘC để reset-safe (đẩy+resume)
EPOCHS  = 30
BATCH   = 128
K       = 3             # sub-centers ArcFace (chịu nhãn nhiễu)
SPLIT   = "page_disjoint"   # hoặc "lobo" (bỏ 1 sách ra test)
HOLDOUT = ""            # ví dụ "stt4" khi SPLIT="lobo"
SAMPLER = "balanced"    # hoặc "confusion" (hard-negative theo chữ giống)
USE_SAM = True          # flat-minima (khuyến nghị)
USE_SWA = False

### 2) Thiết lập môi trường — tự tìm code/data + token HF

In [ ]:
import os, sys, glob, shutil, subprocess
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "huggingface_hub"], check=False)

def find(name, roots=("/kaggle/input", "/kaggle/working", ".")):
    for r in roots:
        hits = sorted(glob.glob(f"{r}/**/{name}", recursive=True))
        if hits:
            return hits[0]
    return None

code_path = find("train.py")
data_path = find("manifest.csv")
assert code_path, "Không thấy ArcFace/train.py — gắn code (dataset hoặc !git clone repo)."
assert data_path, "Không thấy manifest.csv — gắn dataset ArcFace/data (chạy prepare_data.py ở máy trước)."
CODE_DIR, DATA_DIR = os.path.dirname(code_path), os.path.dirname(data_path)
OUT = "/kaggle/working/checkpoints" if os.path.isdir("/kaggle/working") else "checkpoints"

# /kaggle/input read-only -> copy code sang working để chạy được (ghi __pycache__)
if CODE_DIR.startswith("/kaggle/input"):
    dst = "/kaggle/working/ArcFace_code"
    shutil.rmtree(dst, ignore_errors=True); shutil.copytree(CODE_DIR, dst); CODE_DIR = dst
sys.path.insert(0, CODE_DIR)

tok = ""
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    tok = os.environ.get("HF_TOKEN", "")
os.environ["HF_TOKEN"] = tok or ""

import torch
print("code:", CODE_DIR)
print("data:", DATA_DIR)
print("out :", OUT)
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (bật GPU!)")
print("HF  : repo=", HF_REPO or "(none)", "| token=", "yes" if tok else "NO -> local-only, KHÔNG reset-safe")

### 3) Train — reset-safe (chạy lại cell này sau reset để tiếp tục)

In [ ]:
def run(args):
    print("\n$", " ".join(args), flush=True)
    subprocess.run([sys.executable] + args, cwd=CODE_DIR, check=True)

cmd = ["train.py", "--data", DATA_DIR, "--out", OUT,
       "--epochs", str(EPOCHS), "--batch", str(BATCH), "--k", str(K),
       "--sampler", SAMPLER, "--split", SPLIT, "--holdout", HOLDOUT]
if USE_SAM: cmd.append("--sam")
if USE_SWA: cmd.append("--swa")
if HF_REPO: cmd += ["--hf-repo", HF_REPO]
run(cmd)

### 4) Export — gộp sub-center → `best.pt` (drop-in cho infer.py) + đẩy HF

In [ ]:
ecmd = ["export_checkpoint.py", "--data", DATA_DIR,
        "--ckpt", f"{OUT}/train_best.pt", "--out", f"{OUT}/best.pt"]
if HF_REPO: ecmd += ["--hf-repo", HF_REPO]
run(ecmd)

### 5) Đánh giá trung thực (page-disjoint test): retrieval@1/@5 + proxy error-AUC

In [ ]:
run(["evaluate.py", "--data", DATA_DIR, "--ckpt", f"{OUT}/best.pt",
     "--split", SPLIT, "--holdout", HOLDOUT])

### Xong
- `best.pt` ở `/kaggle/working/checkpoints/` **và** trên HF `HF_REPO`.
- Tải về repo:  `huggingface-cli download <HF_REPO> best.pt --local-dir nom-embed`
- Áp **P0/P1** (`inference_s3_v2.score_v2`) vào `visual_signal.decide` → `bash run_pipeline.sh --from build --until publish`.
- **Reset session?** Chạy lại cả notebook — cell **Train** tự resume từ HF, chỉ chạy epoch còn thiếu.